# Modélisation – H5-H6 : pont, taux d'effort, volume et élan
**Contributeurs :** Cécile · **Date :** 12/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`. Les modèles viennent de `gabarit_pont.py` (Elastic Net, apprentissage dès 2017).

**Question :** ajouter le bloc volume et élan (H5) au modèle pont + taux d'effort (H6) améliore-t-il la prévision ?

**Pourquoi ce notebook :** c'est le test fait avant d'ouvrir 2025. Si le bloc apporte quelque chose, ce modèle devient un candidat de plus pour le test final 2025 ; sinon, il reste documenté comme essai non retenu. Le test 2025 n'est **pas** ouvert ici.

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux.
- `TAUX_EFFORT` : le bloc H6 (capacité d'achat, fichier P7).
- `BLOC_H5` : le bloc volume et élan retenu pour H5 (rapport, partie 7.1), issu du fichier P9.
- `LOG_EN_PLUS` : les variables asymétriques passées en log, comme dans H5 et H6, en plus de celles que le pont met déjà en log.
- `COLONNES_P9` : les colonnes de P9 jointes à la table (seules trois d'entre elles entrent dans le modèle ; les quatre servent à écarter les lignes dont un indicateur est vide).
- `SEUIL_GAIN` : seuil de la règle de décision de l'équipe, fixé avant les résultats.
- Les trois modèles, du plus simple au plus complet : pont, pont + taux d'effort, pont + taux d'effort + volume et élan.

Les blocs eux-mêmes sont étudiés dans les notebooks H5 (volume et élan) et H6 (capacité d'achat).

In [6]:
HYPOTHESE = "H5-H6"
SEUIL_GAIN = 3                                       # % de gain sur la MAE, fixé avant les résultats

TAUX_EFFORT = ["taux_effort"]                        # bloc H6
BLOC_H5 = ["nb_ventes_n1", "rotation_parc_type", "evol_prix_n2_n1"]   # bloc H5 retenu (rapport 7.1)
LOG_EN_PLUS = ["taux_effort", "nb_ventes_n1"]        # variables asymétriques passées en log (comme H5 et H6)
COLONNES_P9 = ["rotation_parc", "rotation_parc_type", "evol_ventes_n2_n1", "evol_prix_n2_n1"]

NOM_PONT = f"{HYPOTHESE} – Pont"
NOM_CAPACITE = f"{HYPOTHESE} – Pont + taux d'effort"
NOM_NOUVEAU = f"{HYPOTHESE} – Pont + taux d'effort + volume et élan"

## 1. Import et chargement
La table commune est complétée par deux fichiers de préparation, joints sur l'année, le sous-bassin de vie et le type de bien :
- `fe_volume_elan_sbv.parquet` (P9) : volume et élan ;
- `fe_capacite_achat_sbv.parquet` (P7) : le taux d'effort.

Les clés de jointure sont lues dans la table sans dépendre de la forme de son index. On vérifie que la jointure ne change pas le nombre de lignes.

Pour que les trois modèles soient jugés sur les mêmes lignes, on ne garde que celles dont les indicateurs P9 et le taux d'effort sont renseignés (en plus du masque commun de `charger_table()`). `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec un apprentissage qui commence en 2017 (`ANNEE_DEBUT_PONT`).

In [7]:
import warnings

import pandas as pd

import commun
from commun import CIBLE, charger_table, fenetres, evaluer, tester
from gabarit_pont import ANNEE_DEBUT_PONT, GRILLE_EN, previsions_pont

warnings.filterwarnings("ignore")      # convergence de l'Elastic Net sur les plus petits plis
pd.set_option("display.width", 220)

In [8]:
table_commune = charger_table()

# Clés de jointure lues dans la table, sans dépendre de la forme de son index (comme H5)
cles = pd.DataFrame({"annee": table_commune["annee"].to_numpy(),
                     "sous_bassin_de_vie": table_commune.index.get_level_values("sous_bassin_de_vie"),
                     "type_bien": table_commune.index.get_level_values("type_bien")})

p9 = pd.read_parquet(commun.DATA_OUT / "fe_volume_elan_sbv.parquet")
p7 = pd.read_parquet(commun.DATA_OUT / "fe_capacite_achat_sbv.parquet")[
    ["annee", "sous_bassin_de_vie", "type_bien", "taux_effort"]]

joint = (cles.merge(p9, on=["annee", "sous_bassin_de_vie", "type_bien"], how="left", validate="one_to_one")
             .merge(p7, on=["annee", "sous_bassin_de_vie", "type_bien"], how="left", validate="one_to_one"))
assert len(joint) == len(table_commune), "la jointure a changé le nombre de lignes"

table_jointe = table_commune.copy()
for c in COLONNES_P9 + TAUX_EFFORT:
    table_jointe[c] = joint[c].to_numpy()

# Mêmes lignes pour les trois modèles : indicateurs P9 et taux d'effort renseignés
complet = table_jointe[COLONNES_P9 + TAUX_EFFORT].notna().all(axis=1)
table = table_jointe[complet]
periode = table[table["annee"] >= ANNEE_DEBUT_PONT]
assert (periode["taux_effort"] > 0).all() and (periode["nb_ventes_n1"] > 0).all(), "log impossible"

print("Table commune :", table_commune.shape, "| lignes gardées :", table.shape,
      "| retirées (indicateur vide) :", int((~complet).sum()))
FENETRES = fenetres(table, annee_debut=ANNEE_DEBUT_PONT)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {int(app['annee'].min())}-{int(app['annee'].max())} "
          f"({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

Table commune : (23489, 39) | lignes gardées : (23177, 44) | retirées (indicateur vide) : 312
Fenêtre 2023 : apprentissage 2017-2022 (15369 lignes) | jugée 2023 (2629 lignes)
Fenêtre 2024 : apprentissage 2017-2023 (17998 lignes) | jugée 2024 (2596 lignes)


## 2. Modèles
Les trois modèles passent par le même gabarit (`previsions_pont` de `gabarit_pont.py`) : mêmes réglages, mêmes fenêtres, une prévision par année jugée.

1. **Pont** : le modèle de base du tour 2.
2. **Pont + taux d'effort** : c'est H6 (modèle 2.1).
3. **Pont + taux d'effort + volume et élan** : le nouveau modèle de H5-H6.

Tous les modèles sont gardés, y compris ceux qui perdent. `previsions` garde les prévisions (section 3), `ajustes` les modèles entraînés (section 4).

In [9]:
previsions, ajustes = {}, {}
for nom, bloc in [(NOM_PONT, []),
                  (NOM_CAPACITE, TAUX_EFFORT),
                  (NOM_NOUVEAU, TAUX_EFFORT + BLOC_H5)]:
    modeles = {}
    prev = previsions_pont(table, annee_debut=ANNEE_DEBUT_PONT, bloc=bloc,
                           log_en_plus=[c for c in LOG_EN_PLUS if c in bloc], modeles=modeles)
    for annee, p in prev.items():
        assert p.notna().all(), f"{nom} : prévisions manquantes en {annee}"
        previsions[(nom, annee)] = p
    for cle, m in modeles.items():
        ajustes[(nom,) + cle] = m
    print("Modèle lancé :", nom)

Modèle lancé : H5-H6 – Pont
Modèle lancé : H5-H6 – Pont + taux d'effort
Modèle lancé : H5-H6 – Pont + taux d'effort + volume et élan


## 3. Mesures et tests
Trois étapes, du plus simple au plus exigeant :
1. **Les erreurs** : MAE du modèle et de la référence en €/m², gain sur la référence en %, biais (prévision moins prix réel, en moyenne). C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **Les tests** : ils disent si l'écart avec la référence est plus grand que le hasard.
   - **IC 95 %** (bootstrap par bassin de vie) : la fourchette dans laquelle se trouve l'écart de MAE, en tenant compte du hasard. Si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout) ; autour de 0, pas de différence nette.
3. **La règle de décision** : une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins `SEUIL_GAIN` %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

`mesurer(nom, reference)` fait ces trois étapes pour un modèle. Sans `reference`, la comparaison se fait avec la naïve (prix N = prix N-1). Avec `reference`, la naïve est remplacée par la prévision d'un autre modèle : `evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1`, on leur donne donc une copie de l'année jugée où cette colonne contient la prévision de la référence. Dans les tableaux, la colonne « référence » dit toujours à qui le modèle est comparé.

In [10]:
def mesurer(nom, reference=None):
    """evaluer + tester de commun.py ; reference = nom d'un modèle dont la prévision remplace le prix N-1."""
    res, tst = [], []
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        if reference is not None:
            ev_ref["prix_n1"] = previsions[(reference, annee)]
        evaluer(nom, ev_ref, previsions[(nom, annee)], res)
        tester(nom, ev_ref, previsions[(nom, annee)], tst)
    cles_t = ["modèle", "type de bien", "année jugée"]
    tableau = pd.DataFrame(res).merge(pd.DataFrame(tst), on=cles_t, suffixes=("", " (test)"))
    tableau = tableau.rename(columns=lambda c: c.replace("naïve", "référence"))
    tableau["référence"] = "naive" if reference is None else reference.replace(f"{HYPOTHESE} – ", "").lower()
    # Biais : prévision moins prix réel, moyenne par type de bien
    biais = []
    for _, r in tableau.iterrows():
        ev = FENETRES[r["année jugée"]][1]
        idx = ev.index[ev.index.get_level_values("type_bien") == r["type de bien"]]
        biais.append((previsions[(nom, r["année jugée"])].loc[idx] - ev.loc[idx, CIBLE]).mean())
    tableau["biais (€/m²)"] = biais
    tableau["année réussie"] = ((tableau["IC 95 % haut (€/m²)"] < 0) & (tableau["rang-bisériale"] < 0)
                                & (tableau["gain sur la référence (%)"] >= SEUIL_GAIN))
    return tableau


COLONNES = ["modèle", "référence", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
            "gain sur la référence (%)", "IC 95 % haut (€/m²)", "rang-bisériale", "biais (€/m²)", "année réussie"]

### 3.1 Chaque modèle contre la naïve
La naïve (prix N = prix N-1) est la référence commune. Chaque modèle doit d'abord la battre.

In [11]:
tableaux = [mesurer(NOM_PONT), mesurer(NOM_CAPACITE), mesurer(NOM_NOUVEAU),
            mesurer(NOM_NOUVEAU, reference=NOM_CAPACITE),       # question de l'hypothèse
            mesurer(NOM_NOUVEAU, reference=NOM_PONT)]
resultats = pd.concat(tableaux, ignore_index=True)

contre_naive = resultats[resultats["référence"] == "naive"]
contre_naive[COLONNES].round(2)

,modèle,référence,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),IC 95 % haut (€/m²),rang-bisériale,biais (€/m²),année réussie
0,H5-H6 – Pont,naive,appartement,2023,159.89,166.02,3.69,10.88,-0.07,45.48,False
1,H5-H6 – Pont,naive,maison,2023,130.05,111.76,-16.36,23.21,0.25,71.50,False
2,H5-H6 – Pont,naive,appartement,2024,189.41,147.53,-28.39,48.55,0.48,138.60,False
3,H5-H6 – Pont,naive,maison,2024,179.68,129.19,-39.08,54.37,0.68,149.99,False
4,H5-H6 – Pont + taux d'effort,naive,appartement,2023,167.25,166.02,-0.74,25.53,-0.10,13.43,False
5,H5-H6 – Pont + taux d'effort,naive,maison,2023,123.36,111.76,-10.38,15.63,0.15,36.38,False
6,H5-H6 – Pont + taux d'effort,naive,appartement,2024,165.13,147.53,-11.93,22.57,0.30,91.75,False
7,H5-H6 – Pont + taux d'effort,naive,maison,2024,125.14,129.19,3.13,-0.78,-0.17,48.28,True
8,H5-H6 – Pont + taux d'effort + volume et élan,naive,appartement,2023,162.35,166.02,2.21,18.78,-0.07,50.58,False
9,H5-H6 – Pont + taux d'effort + volume et élan,naive,maison,2023,119.96,111.76,-7.33,12.52,0.09,36.72,False


### 3.2 Le volume et l'élan apportent-ils quelque chose ?
La question de l'hypothèse : le nouveau modèle fait-il mieux que le pont + taux d'effort (H6), et mieux que le pont seul ? Mêmes mesures, la naïve est remplacée par le modèle de comparaison.

In [12]:
contre_modeles = resultats[resultats["référence"] != "naive"]
contre_modeles[COLONNES].round(2)

,modèle,référence,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),IC 95 % haut (€/m²),rang-bisériale,biais (€/m²),année réussie
12,H5-H6 – Pont + taux d'effort + volume et élan,pont + taux d'effort,appartement,2023,162.35,167.25,2.93,1.13,-0.02,50.58,False
13,H5-H6 – Pont + taux d'effort + volume et élan,pont + taux d'effort,maison,2023,119.96,123.36,2.76,0.03,-0.03,36.72,False
14,H5-H6 – Pont + taux d'effort + volume et élan,pont + taux d'effort,appartement,2024,188.38,165.13,-14.08,27.97,0.45,137.94,False
15,H5-H6 – Pont + taux d'effort + volume et élan,pont + taux d'effort,maison,2024,133.25,125.14,-6.48,11.22,0.16,83.48,False
16,H5-H6 – Pont + taux d'effort + volume et élan,pont,appartement,2023,162.35,159.89,-1.54,9.07,0.01,50.58,False
17,H5-H6 – Pont + taux d'effort + volume et élan,pont,maison,2023,119.96,130.05,7.76,-6.18,-0.21,36.72,True
18,H5-H6 – Pont + taux d'effort + volume et élan,pont,appartement,2024,188.38,189.41,0.54,3.07,0.01,137.94,False
19,H5-H6 – Pont + taux d'effort + volume et élan,pont,maison,2024,133.25,179.68,25.84,-42.55,-0.65,83.48,True


### 3.3 Règle de décision
Une ligne par modèle, par référence et par type de bien : nombre d'années réussies sur les années jugées, et décision.

In [13]:
decision = (resultats.groupby(["modèle", "référence", "type de bien"], sort=False)["année réussie"]
            .agg(["sum", "count"]).rename(columns={"sum": "années réussies", "count": "années jugées"}))
decision["retenu"] = decision["années réussies"] > decision["années jugées"] / 2
decision

années réussies  années jugées  retenu
modèle                                        référence            type de bien                                        
H5-H6 – Pont                                  naive                appartement                 0              2   False
                                                                   maison                      0              2   False
H5-H6 – Pont + taux d'effort                  naive                appartement                 0              2   False
                                                                   maison                      1              2   False
H5-H6 – Pont + taux d'effort + volume et élan naive                appartement                 0              2   False
                                                                   maison                      0              2   False
                                              pont + taux d'effort appartement                 0              2   False
                                                                   maison                      0              2   False
                                              pont                 appartement                 0              2   False
                                                                   maison                      2              2    True

**Comment lire le tableau de décision :** prenons « Pont + taux d'effort, contre la naïve, maisons ». Il réussit 1 année sur 2 : 2024 (gain de 3,13 %, IC 95 % haut à −0,78 €/m², rang-bisériale −0,17), mais pas 2023 (gain de −10,38 %). Il faut la majorité des années, soit 2 sur 2 ici : il n'est pas retenu.

**Résultats contre la naïve :** aucun des trois modèles n'est retenu, ni pour les appartements ni pour les maisons. Le pont seul fait nettement moins bien que la naïve en 2024 (−28,39 % pour les appartements, −39,08 % pour les maisons). Tous les modèles surestiment les prix (biais positif, jusqu'à +150 €/m² pour le pont, maisons 2024).

**Résultats du nouveau modèle contre le pont + taux d'effort, puis contre le pont seul :** contre le pont + taux d'effort, le volume et l'élan n'apportent rien : aucune année réussie. En 2023 le gain est faible (2,93 % pour les appartements, 2,76 % pour les maisons) et l'IC reste au-dessus de 0 (1,13 et 0,03 €/m²) ; en 2024 le nouveau modèle fait moins bien (−14,08 % et −6,48 %). Contre le pont seul, le tableau indique « retenu » pour les maisons (gains de 7,76 % et 25,84 %), mais cet écart vient surtout du taux d'effort : MAE des maisons en 2024 de 179,68 €/m² (pont) à 125,14 (avec le taux d'effort), puis 133,25 avec le volume et l'élan en plus.

## 4. Interprétation

### 4.1 Contrôle du réglage
L'Elastic Net choisit `alpha` et `l1_ratio` sur les plis temporels, pour chaque année jugée et chaque type de bien. Si `alpha` tombe à une borne de la grille (`GRILLE_EN`), la grille est peut-être trop étroite : le meilleur réglage serait au-delà. Le contrôle signale ces cas.

In [14]:
bornes = (min(GRILLE_EN["alpha"]), max(GRILLE_EN["alpha"]))
for (nom, annee, type_bien), m in ajustes.items():
    alpha = m.reglage_["alpha"]
    bord = "  <- au bord de la grille" if alpha in bornes else ""
    print(f"{nom} | {annee} | {type_bien} : alpha = {alpha:.1e}, l1_ratio = {m.reglage_['l1_ratio']}{bord}")

H5-H6 – Pont | 2023 | appartement : alpha = 3.2e-03, l1_ratio = 0.8
H5-H6 – Pont | 2023 | maison : alpha = 1.0e-03, l1_ratio = 0.8
H5-H6 – Pont | 2024 | appartement : alpha = 3.2e-03, l1_ratio = 0.8
H5-H6 – Pont | 2024 | maison : alpha = 3.2e-04, l1_ratio = 0.8
H5-H6 – Pont + taux d'effort | 2023 | appartement : alpha = 3.2e-04, l1_ratio = 0.8
H5-H6 – Pont + taux d'effort | 2023 | maison : alpha = 1.0e-05, l1_ratio = 0.2  <- au bord de la grille
H5-H6 – Pont + taux d'effort | 2024 | appartement : alpha = 1.0e-03, l1_ratio = 0.8
H5-H6 – Pont + taux d'effort | 2024 | maison : alpha = 1.0e-05, l1_ratio = 0.2  <- au bord de la grille
H5-H6 – Pont + taux d'effort + volume et élan | 2023 | appartement : alpha = 3.2e-03, l1_ratio = 0.8
H5-H6 – Pont + taux d'effort + volume et élan | 2023 | maison : alpha = 1.0e-05, l1_ratio = 0.2  <- au bord de la grille
H5-H6 – Pont + taux d'effort + volume et élan | 2024 | appartement : alpha = 3.2e-03, l1_ratio = 0.8
H5-H6 – Pont + taux d'effort + volume e

### 4.2 Poids du bloc dans le nouveau modèle
Coefficients standardisés du nouveau modèle, par année jugée et type de bien : les variables étant à la même échelle, leurs coefficients se comparent. Un coefficient proche de 0 veut dire que le modèle ne se sert presque pas de la variable ; l'Elastic Net peut aussi le ramener exactement à 0.

In [15]:
coefs = pd.DataFrame({(a, tb): m.coefs_[["prix_n1"] + TAUX_EFFORT + BLOC_H5]
                      for (nom, a, tb), m in ajustes.items() if nom == NOM_NOUVEAU})
coefs.round(4)

2023                2024        
                   appartement  maison appartement  maison
prix_n1                 0.5146  0.5062      0.5172  0.5149
taux_effort            -0.0000 -0.0597      0.0000 -0.0644
nb_ventes_n1            0.0000 -0.0258      0.0000 -0.0224
rotation_parc_type      0.0197  0.0319      0.0173  0.0274
evol_prix_n2_n1        -0.0130 -0.0244     -0.0151 -0.0254

## 5. Conclusion
- Contre la naïve : aucun modèle retenu. Meilleur cas : pont + taux d'effort, maisons 2024 (gain de 3,13 %, IC sous 0), mais pas en 2023.
- Le volume et l'élan n'améliorent pas le pont + taux d'effort : 0 année réussie sur 4 cas, et une dégradation en 2024. L'avantage contre le pont seul vient du taux d'effort.
- Coefficients : pour les appartements, le taux d'effort et le nombre de ventes N-1 sont ramenés à 0 ; pour les maisons, le taux d'effort pèse environ −0,06.
- Réglage : alpha tombe à la borne basse de la grille pour les maisons (modèles avec taux d'effort), sans effet probable sur la conclusion, la pénalité étant déjà presque nulle.
- Candidat pour le test final 2025 : le modèle hiérarchique à ordonnée aléatoire, avec les variables du pont + taux d'effort (H6, modèle 2.2). H5-H6 ne justifie pas d'y ajouter le volume et l'élan (testé avec l'Elastic Net du pont, pas avec le hiérarchique).
- Limites : un seul gabarit (Elastic Net), deux années jugées, tous les modèles surestiment les prix en 2024.